# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's enumerate all record sets and their associated fields. (Referencing Croissant entities by their `@id` only.)

In [ ]:
# List available record sets and their fields, by their @id
record_sets = dataset.record_sets()

print("Available Record Sets:")
overview = {}
for rs in record_sets:
    print(f"- Record Set @id: {rs['@id']}, name: {rs.get('name', '')}")
    overview[rs['@id']] = []
    if 'field' in rs:
        print("   Fields:")
        for field in rs['field']:
            # Each field is a dict with @id and name
            print(f"     - Field @id: {field['@id']}, name: {field.get('name', '')}")
            overview[rs['@id']].append(field['@id'])
    print()
# Let's also display a DataFrame summary
overview_df = pd.DataFrame([
    {'record_set': rs, 'field': f}
    for rs, fields in overview.items() for f in fields
])
overview_df

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Based on the above, select main survey data record set.
# Adjust the @id as needed based on the record sets printed above.
# For demonstration, we extract all record sets available.
main_record_set_id = None
for rs in dataset.record_sets():
    if 'survey' in rs.get('name', '').lower() or 'main' in rs.get('name', '').lower():
        main_record_set_id = rs['@id']
        break
if not main_record_set_id:
    # Pick first as fallback
    main_record_set_id = dataset.record_sets()[0]['@id']

# List all available record sets' @id
record_sets_ids = [rs['@id'] for rs in dataset.record_sets()]
print(f"Record set IDs: {record_sets_ids}")

dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading data for record set {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records. Fields: {df.columns.tolist()}")
    print()
# Show the first few rows for the main record set
print(f"--- First five rows for {main_record_set_id} ---")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we select numerical fields, filter on a scoring variable, and show how to group and normalize.

In [ ]:
import numpy as np
df = dataframes[main_record_set_id]

# Choose a numeric field by @id (example: PHQ-9 score). Adjust if your dataset has a different field @id.
candidate_numeric_fields = [col for col in df.columns if (('score' in col.lower()) or (df[col].dtype in [float, int, np.int64, np.float64]))]
print(f"Detected numeric fields: {candidate_numeric_fields}")
# For this dataset, GAD-7, PHQ-9, or PSQ total scores are likely present. Choose phq9_total if available.
if 'phq9_total' in df.columns:
    numeric_field = 'phq9_total'
elif len(candidate_numeric_fields) > 0:
    numeric_field = candidate_numeric_fields[0]
else:
    raise ValueError('No obvious numeric field to analyze.')

# Filter, e.g., show respondents with phq9_total > 10 (moderate/severe depression)
threshold = 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the selected field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical variable, e.g. gender or education level. Pick available columns smartly.
for preferred_group in ['gender', 'sex', 'level_of_education', 'village']:
    if preferred_group in df.columns:
        group_field = preferred_group
        print(f"Selected group field: {group_field}")
        break
else:
    group_field = df.columns[0]  # fallback

if group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Mean {numeric_field} by {group_field} for filtered group:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(8, 5))
# Histogram of the numeric field
sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Barplot of the group means if categorical
if group_field in df.columns:
    plt.figure(figsize=(8,5))
    order = filtered_df[group_field].value_counts().index
    sns.barplot(x=group_field, y=numeric_field, data=filtered_df, order=order, ci=None)
    plt.title(f"Mean {numeric_field} by {group_field} (> {threshold})")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we demonstrated loading a FAIR² Croissant mental health survey dataset using `mlcroissant`.
* We explored the dataset structure via record set and field `@id`s, loaded survey responses, and conducted exploratory data analysis, such as filtering and group means by categorical attributes.
* Visualizations illustrated both overall score distributions and group differences for further public health research in Kilifi County, Kenya.